In [28]:
!nvidia-smi -L

GPU 0: NVIDIA GeForce RTX 2080 Ti (UUID: GPU-55eaad7d-8fd8-5cf6-a56c-eb8ca7aeb6da)


In [29]:
%pip show qiskit qiskit-aer qiskit-aer-gpu

Name: qiskit
Version: 2.3.1
Summary: An open-source SDK for working with quantum computers at the level of extended quantum circuits, operators, and primitives.
Home-page: https://www.ibm.com/quantum/qiskit
Author: 
Author-email: Qiskit Development Team <qiskit@us.ibm.com>
License-Expression: Apache-2.0
Location: /usr/users/st7eco/nompai_gae/Eco-QNN/.venv/lib/python3.10/site-packages
Requires: dill, numpy, rustworkx, scipy, stevedore, typing-extensions
Required-by: ibm-quantum-schemas, qiskit-aer, qiskit-algorithms, qiskit-ibm-runtime, qiskit-machine-learning, samplomatic
---
Name: qiskit-aer
Version: 0.17.2
Summary: Aer - High performance simulators for Qiskit
Home-page: https://github.com/Qiskit/qiskit-aer
Author: AER Development Team
Author-email: qiskit@us.ibm.com
License: Apache 2.0
Location: /usr/users/st7eco/nompai_gae/Eco-QNN/.venv/lib/python3.10/site-packages
Requires: numpy, psutil, python-dateutil, qiskit, scipy
Required-by: 
Note: you may need to restart the kernel to use u

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_ibm_runtime import SamplerV2 as Sampler
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize
from qiskit_aer import StatevectorSimulator

In [20]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Dataset
n          = 1000    # Number of generated points
TEST_SIZE  = 0.2    # Fraction used for test set
SEED       = 1      # Random seed (data split & optimizer init)
N_DIM      = 3    # Features dimension

# Optimization
MAXITER    = 200     # minimizer max iterations
optimizer = COBYLA(maxiter=MAXITER) 

# Noise 
USE_NOISE = True 
NOISE_RATE = 0.02
# ──────────────────────────────────────────────────────────────────────────────
REUPLOAD_SHOTS = 1024
RC_REUPLOAD = 8
MAXITER_REUPLOAD = 200
USE_REUPLOAD = (N_DIM == 3)

print(f"USE_REUPLOAD = {USE_REUPLOAD}")


USE_REUPLOAD = True


In [21]:
if USE_NOISE:
    nm = NoiseModel()
    err1 = depolarizing_error(NOISE_RATE, 1)
    nm.add_all_qubit_quantum_error(err1, ['u', 'h', 'ry', 'rz'])
    err2 = depolarizing_error(NOISE_RATE * 5, 2)
    nm.add_all_qubit_quantum_error(err2, ['cx'])
    backend = AerSimulator(noise_model=nm, method="density_matrix", device="GPU")
    sampler = Sampler(backend)
else:
    sampler = Sampler(AerSimulator(device="GPU"))

In [22]:
def generate_nsphere_data(n_samples, n_dim, radius=None, seed=None):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, (n_samples, n_dim))
    if radius is None:
        radius = np.sqrt(n_dim / 3)
    dist_sq = np.sum(X**2, axis=1)
    y = (dist_sq >= radius**2).astype(int)
    return X, y, radius

In [23]:
X, y, R = generate_nsphere_data(n_samples=n, n_dim=N_DIM, seed=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)
print(X_train.shape, y_train.shape)

(800, 3) (800,)


In [24]:
def U_su2_reupload(qc, theta_block, omega_block, x, qubit=0):
    """
    Une couche de type U(theta + omega * x) sur un seul qubit.
    Pour N_DIM=3, on injecte directement les 3 coordonnées.
    """
    qc.u(
        theta_block[0] + omega_block[0] * x[0],
        theta_block[1] + omega_block[1] * x[1],
        theta_block[2] + omega_block[2] * x[2],
        qubit
    )

def create_reupload_circuit(x, theta, omega, measure=False):
    qc = QuantumCircuit(1)
    for r in range(RC_REUPLOAD):
        qc.h(0)
        U_su2_reupload(qc, theta[r], omega[r], x, qubit=0)
    if measure:
        qc.measure_all()
    return qc

def get_reupload_probs_batch(circuits, shots=REUPLOAD_SHOTS):
    """
    - Si USE_NOISE = True : exécute les circuits mesurés sur AerSimulator bruité.
    - Sinon : utilise StatevectorSimulator pour obtenir les probabilités exactes.
    """
    probs = []

    if USE_NOISE:
        measured_circuits = []
        for qc in circuits:
            qc_m = qc.copy()
            if qc_m.num_clbits == 0:
                qc_m.measure_all()
            measured_circuits.append(qc_m)

        transpiled_circuits = transpile(measured_circuits, backend)
        result = backend.run(transpiled_circuits, shots=shots).result()

        for qc_t in transpiled_circuits:
            counts = result.get_counts(qc_t)
            p0 = counts.get('0', 0) / shots
            p1 = counts.get('1', 0) / shots
            probs.append((p0, p1))
    else:
        sim = StatevectorSimulator()
        for qc in circuits:
            sv = sim.run(qc).result().get_statevector()
            p0 = float(np.abs(sv[0])**2)
            p1 = float(np.abs(sv[1])**2)
            probs.append((p0, p1))

    return probs


In [25]:
def predict_reupload_batch(X, theta, omega):
    circuits = [create_reupload_circuit(x, theta, omega, measure=False) for x in X]
    probs = get_reupload_probs_batch(circuits)
    return np.array([0 if p0 >= p1 else 1 for p0, p1 in probs])

def evaluate_binary_predictions(y_true, y_pred, positive_label=1):
    tp = np.sum((y_pred == positive_label) & (y_true == positive_label))
    fp = np.sum((y_pred == positive_label) & (y_true != positive_label))
    fn = np.sum((y_pred != positive_label) & (y_true == positive_label))
    tn = np.sum((y_pred != positive_label) & (y_true != positive_label))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    accuracy = np.mean(y_pred == y_true)

    return {
        "precision": precision,
        "accuracy": accuracy,
        "recall": recall,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }

def evaluate_reupload(X, y, theta, omega, positive_label=1):
    y_pred = predict_reupload_batch(X, theta, omega)
    metrics = evaluate_binary_predictions(y, y_pred, positive_label=positive_label)
    metrics["y_pred"] = y_pred
    return metrics

In [26]:
N_ESTIMATORS_REUPLOAD = 5
MAXITER_WEAK_REUPLOAD = MAXITER_REUPLOAD

def reupload_cost_weighted_samples(params, X, y, sample_weights):
    theta = params[:3 * RC_REUPLOAD].reshape(RC_REUPLOAD, 3)
    omega = params[3 * RC_REUPLOAD:].reshape(RC_REUPLOAD, 3)

    circuits = [create_reupload_circuit(x, theta, omega, measure=False) for x in X]
    probs = get_reupload_probs_batch(circuits)

    total_cost = 0.0
    for i, y_target in enumerate(y):
        p0, p1 = probs[i]
        y_expected = (1.0, 0.0) if y_target == 0 else (0.0, 1.0)
        total_cost += sample_weights[i] * ((p0 - y_expected[0])**2 + (p1 - y_expected[1])**2)

    return 0.5 * total_cost / np.sum(sample_weights)

def optimize_reupload_weak_learner(X, y, sample_weights, seed):
    rng = np.random.default_rng(seed)
    init = rng.uniform(-np.pi, np.pi, size=6 * RC_REUPLOAD)

    res = minimize(
        reupload_cost_weighted_samples,
        init,
        args=(X, y, sample_weights),
        method="COBYLA",
        options={"maxiter": MAXITER_WEAK_REUPLOAD},
    )

    theta = res.x[:3 * RC_REUPLOAD].reshape(RC_REUPLOAD, 3)
    omega = res.x[3 * RC_REUPLOAD:].reshape(RC_REUPLOAD, 3)
    return theta, omega, res.fun

def train_adaboost_reupload(X_train, y_train, n_estimators=N_ESTIMATORS_REUPLOAD):
    n = len(y_train)
    weights = np.ones(n) / n
    estimators = []

    for i in range(n_estimators):
        theta, omega, cost = optimize_reupload_weak_learner(
            X_train, y_train, weights, seed=SEED + 100 + i
        )

        y_pred = predict_reupload_batch(X_train, theta, omega)
        incorrect = (y_pred != y_train).astype(float)
        eps = np.dot(weights, incorrect)
        eps = np.clip(eps, 1e-10, 1 - 1e-10)

        alpha = 0.5 * np.log((1 - eps) / eps)

        y_signed = np.where(y_train == 0, -1.0, 1.0)
        h_signed = np.where(y_pred == 0, -1.0, 1.0)
        weights *= np.exp(-alpha * y_signed * h_signed)
        weights /= np.sum(weights)

        estimators.append((theta, omega, alpha))
        print(f"round {i + 1:02d} | cost={cost:.4f} eps={eps:.4f} alpha={alpha:.4f}")

    return estimators

def predict_adaboost_reupload(X, estimators):
    scores = np.zeros(len(X))
    for theta, omega, alpha in estimators:
        h_signed = np.where(predict_reupload_batch(X, theta, omega) == 0, -1.0, 1.0)
        scores += alpha * h_signed
    return np.where(scores >= 0, 1, 0)

def evaluate_adaboost_reupload(X, y, estimators, positive_label=1):
    y_pred = predict_adaboost_reupload(X, estimators)
    metrics = evaluate_binary_predictions(y, y_pred, positive_label=positive_label)
    metrics["y_pred"] = y_pred
    return metrics


In [27]:
# Study: Precision vs Number of iteration of adaboost
layers_list = [1, 2, 4, 6, 10]
precisions = []
accuracies = []

for L in layers_list:
    print(f"\n--- Testing with {L} adaboost iterations ---")
    global N_ESTIMATORS_REUPLOAD
    N_ESTIMATORS_REUPLOAD = L
    estimators_reupload_boost = train_adaboost_reupload(
        X_train, y_train, n_estimators=N_ESTIMATORS_REUPLOAD
    )
    
    metrics_reupload_boost = evaluate_adaboost_reupload(X_test, y_test, estimators_reupload_boost)
    y_pred_reupload_boost = metrics_reupload_boost["y_pred"]
    reupload_boost_accuracy = metrics_reupload_boost["accuracy"]
    precisions.append(metrics_reupload_boost["precision"])
    accuracies.append(reupload_boost_accuracy)
    print(f"Iterations: {L} | Precision: {metrics_reupload_boost['precision']:.4f} | Accuracy: {reupload_boost_accuracy:.4f}")

# Plotting the results
plt.figure(figsize=(10, 6))
plt.plot(layers_list, precisions, marker='o', label='Precision')
plt.plot(layers_list, accuracies, marker='s', linestyle='--', label='Accuracy')
plt.xlabel('Number of Adaboost Iterations')
plt.ylabel('Metric Value')
plt.title('Performance vs Number of Adaboost Iterations')
plt.grid(True)
plt.legend()
plt.show()



--- Testing with 1 adaboost iterations ---


RuntimeError: Simulation device "GPU" is not supported on this system